In [1]:
BRONZE_PATH = "Files/bronze/online_retail_II.csv"
SILVER_PATH = "Files/silver/retail_transactions"
 
print("Paths configured. Ready to read bronze data.")


StatementMeta(, 8b27001d-64b6-4a27-a362-f560017c866d, 3, Finished, Available, Finished, False)

Paths configured. Ready to read bronze data.


In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DecimalType
 
df_bronze = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(BRONZE_PATH))
 
print(f"Row count (raw): {df_bronze.count()}")
df_bronze.printSchema()


StatementMeta(, 8b27001d-64b6-4a27-a362-f560017c866d, 4, Finished, Available, Finished, False)

Row count (raw): 1048575
root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: integer (nullable = true)
 |-- Country: string (nullable = true)



In [3]:
df_clean = df_bronze \
    .filter(F.col("Customer ID").isNotNull()) \
    .filter(~F.col("Invoice").startswith("C")) \
    .filter((F.col("Quantity") > 0) & (F.col("Price") > 0))
 
print(f"Rows after null CustomerID filter: after removing nulls")
print(f"Rows after removing cancellations (C prefix): reduced")
print(f"Rows after removing negative quantities/prices: clean")


StatementMeta(, 8b27001d-64b6-4a27-a362-f560017c866d, 5, Finished, Available, Finished, False)

Rows after null CustomerID filter: after removing nulls
Rows after removing cancellations (C prefix): reduced
Rows after removing negative quantities/prices: clean


In [10]:
df_silver = (df_clean
    .withColumn("InvoiceDate",
        F.to_timestamp(F.col("InvoiceDate"), "dd-MM-yyyy HH:mm"))
    .withColumn("Customer ID",
        F.col("Customer ID").cast(IntegerType()))
    .withColumn("Price",
        F.col("Price").cast(DecimalType(10, 2)))
    .withColumn("Quantity",
        F.col("Quantity").cast(IntegerType()))
    .withColumn("LineTotal",
        F.round(F.col("Quantity") * F.col("Price"), 2))
    .withColumn("ingestion_ts", F.current_timestamp())
    .dropDuplicates(["Invoice", "StockCode"]))

df_silver = df_silver.withColumnRenamed("Customer ID", "Customer_ID")

 
print(f"Silver row count: {df_silver.count()}")


StatementMeta(, 8b27001d-64b6-4a27-a362-f560017c866d, 12, Finished, Available, Finished, False)

Silver row count: 757038


In [11]:
df_silver.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .partitionBy("Country") \
    .save(SILVER_PATH)
 
print("Bronze → Silver complete. Delta table written to Files/silver/")


StatementMeta(, 8b27001d-64b6-4a27-a362-f560017c866d, 13, Finished, Available, Finished, False)

Bronze → Silver complete. Delta table written to Files/silver/
